# The TabPFN Dimensionality Wall (Group 3.3): 
> Track average accuracy across datasetsfiltered by d < 50, d = 100, and d > 100. This isolates your main tuning-free competitor,demonstrating its hard architectural limits while SmallMLP scales gracefully.



In [ ]:
import os
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())
print("Token loaded:", bool(os.getenv("TABPFN_TOKEN")))

In [1]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNClassifier
import xgboost as xgb

/Users/mac/Desktop/SmallData_paper/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from smallgbm import SmallGBMClassifier
from smallmlp import SmallMLPClassifier

In [3]:
warnings.filterwarnings("ignore")

# 1. Load base real dataset (n=569, d=30) and inject noise features to test the d=100 wall
X_base, y = load_breast_cancer(return_X_y=True)
n_samples, d_base = X_base.shape

target_dims = [30, 50, 80, 100, 120, 150, 200, 300]
results = {
    "SmallMLP": [],
    "SmallGBM": [],
    "TabPFN (PCA-100 >100d)": [],
    "XGBoost (Default)": [],
    "Random Forest": [],
}
stds = {k: [] for k in results}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for d_target in target_dims:
    rng = np.random.RandomState(42 + d_target)
    if d_target > d_base:
        noise = rng.normal(0, 1, size=(n_samples, d_target - d_base))
        X = np.hstack([X_base, noise])
        # Shuffle column order so truncation/subsampling isn't trivially seeing clean features first
        perm = rng.permutation(d_target)
        X = X[:, perm]
    else:
        X = X_base.copy()

    fold_scores = {k: [] for k in results}

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        # 1. SmallMLP (frozen alpha=6.0, beta=0.7)
        mlp = SmallMLPClassifier(alpha=6.0, beta=0.7, random_state=42)
        mlp.fit(X_tr_s, y_tr)
        fold_scores["SmallMLP"].append(accuracy_score(y_te, mlp.predict(X_te_s)))

        # 2. SmallGBM
        sgbm = SmallGBMClassifier(random_state=42)
        sgbm.fit(X_tr, y_tr)
        fold_scores["SmallGBM"].append(accuracy_score(y_te, sgbm.predict(X_te)))

        # 3. TabPFN (natively <= 100d; apply standard PCA-100 projection when d > 100)
        tabpfn = TabPFNClassifier(device="cpu")
        if d_target > 100:
            pca = PCA(n_components=100, random_state=42)
            X_tr_pfn = pca.fit_transform(X_tr_s)
            X_te_pfn = pca.transform(X_te_s)
        else:
            X_tr_pfn, X_te_pfn = X_tr_s, X_te_s
        tabpfn.fit(X_tr_pfn, y_tr)
        fold_scores["TabPFN (PCA-100 >100d)"].append(
            accuracy_score(y_te, tabpfn.predict(X_te_pfn))
        )

        # 4. XGBoost Default
        model_xgb = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
        model_xgb.fit(X_tr, y_tr)
        fold_scores["XGBoost (Default)"].append(
            accuracy_score(y_te, model_xgb.predict(X_te))
        )

        # 5. Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_tr, y_tr)
        fold_scores["Random Forest"].append(
            accuracy_score(y_te, rf.predict(X_te))
        )

    for k in results:
        results[k].append(np.mean(fold_scores[k]))
        stds[k].append(np.std(fold_scores[k]))
    print(
        f"d={d_target:3d} | SmallMLP: {results['SmallMLP'][-1]:.4f} | "
        f"SmallGBM: {results['SmallGBM'][-1]:.4f} | TabPFN: {results['TabPFN (PCA-100 >100d)'][-1]:.4f}"
    )

# 2. Save raw table for LaTeX
df_out = pd.DataFrame(results, index=target_dims)
df_out.index.name = "Dimensionality (d)"
df_out.to_csv("tabpfn_dim_wall_results.csv")

# 3. Publication-grade plot (no top title, clean LaTeX fonts)
plt.figure(figsize=(7.5, 4.8), dpi=300)

styles = {
    "SmallMLP": ("#1f77b4", "o", "-", 2.5),
    "SmallGBM": ("#2ca02c", "s", "-", 2.2),
    "TabPFN (PCA-100 >100d)": ("#d62728", "^", "--", 2.2),
    "XGBoost (Default)": ("#7f7f7f", "d", ":", 1.8),
    "Random Forest": ("#ff7f0e", "v", "-.", 1.8),
}

for name, (color, marker, ls, lw) in styles.items():
    means = np.array(results[name])
    err = np.array(stds[name])
    plt.plot(
        target_dims,
        means,
        label=name,
        color=color,
        marker=marker,
        linestyle=ls,
        linewidth=lw,
        markersize=6,
    )
    if name in ["SmallMLP", "TabPFN (PCA-100 >100d)"]:
        plt.fill_between(
            target_dims, means - err, means + err, color=color, alpha=0.12
        )

# Highlight the d=100 architectural boundary
plt.axvline(x=100, color="#d62728", linestyle=":", linewidth=1.8, alpha=0.8)
plt.text(
    104,
    plt.ylim()[0] + 0.015,
    "TabPFN Native Limit ($d=100$)\n(PCA Projection Regime $\\rightarrow$)",
    color="#d62728",
    fontsize=10,
    fontweight="bold",
)

plt.xlabel(r"Feature Dimensionality ($d$)", fontsize=12, labelpad=8)
plt.ylabel("5-Fold Test Accuracy", fontsize=12, labelpad=8)
plt.xticks(target_dims, fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend(frameon=True, fontsize=9.5, loc="lower left")

plt.tight_layout()
plt.savefig("tabpfn_dim_wall.pdf", bbox_inches="tight")
plt.savefig("tabpfn_dim_wall.png", dpi=300, bbox_inches="tight")
plt.show()

TabPFNLicenseError: TabPFN requires a one-time license acceptance to download
model weights for local inference, but no interactive terminal
is available.

To authenticate in a non-interactive environment:
  1. Open https://ux.priorlabs.ai in a browser and log in (or register)
  2. Accept the license on the Licenses tab
  3. Copy your API Key from https://ux.priorlabs.ai/account
  4. Set the environment variable: export TABPFN_TOKEN="<your-api-key>"
     or in Python (before calling .fit()): import os; os.environ["TABPFN_TOKEN"] = "<your-api-key>"